<a href="https://colab.research.google.com/github/lelongc/rac/blob/main/pod.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# @title ⚙️ BƯỚC 1: CÀI ĐẶT THƯ VIỆN & TẢI MODEL AI
import os
from IPython.display import Audio, display, clear_output

print("⏳ Đang cài đặt thư viện (Chỉ mất 1-2 phút)...")
os.system('pip install -U qwen-tts huggingface_hub pydub')
os.system('apt-get install -y ffmpeg sox libsox-fmt-all')
clear_output()
print("✅ Cài đặt xong thư viện!")

from qwen_tts import Qwen3TTSModel
import torch
import soundfile as sf
import gc
import re
from pydub import AudioSegment

torch.backends.cudnn.benchmark = True
current_model = None
current_model_type = None

def load_model(task_type):
    global current_model, current_model_type
    model_name = "Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign" if task_type == "DESIGN" else "Qwen/Qwen3-TTS-12Hz-1.7B-Base"

    if current_model_type == task_type and current_model is not None:
        return current_model

    if current_model:
        del current_model
        gc.collect()
        torch.cuda.empty_cache()

    print(f"📥 Đang tải Model {task_type}... (Vui lòng đợi)")
    current_model = Qwen3TTSModel.from_pretrained(model_name, torch_dtype=torch.float16, device_map="cuda:0", attn_implementation="sdpa")
    current_model_type = task_type
    return current_model

print("✅ Hệ thống đã sẵn sàng!")

✅ Cài đặt xong thư viện!

********
********
 
✅ Hệ thống đã sẵn sàng!


/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


In [ ]:
# @title 🎙️ BƯỚC 2: TẠO GIỌNG MẪU NAM & NỮ

# --- BẠN CÓ THỂ SỬA TEXT Ở ĐÂY HOẶC GIỮ NGUYÊN ---
nam_prompt = "A professional male podcast host, deep and soothing voice. Laughing and energetic."
nam_text = "Haha! Hello everyone, welcome to the show."

nu_prompt = "A warm female voice, soft and velvety. American accent. Very happy."
nu_text = "Wow! I am so excited to be here."
# --------------------------------------------------

print("⏳ Đang tạo giọng Nam...")
model = load_model("DESIGN")
with torch.inference_mode():
    w_nam, sr_nam = model.generate_voice_design(text=nam_text, instruct=nam_prompt)
sf.write("nam_ref.wav", w_nam[0], sr_nam)

print("⏳ Đang tạo giọng Nữ...")
with torch.inference_mode():
    w_nu, sr_nu = model.generate_voice_design(text=nu_text, instruct=nu_prompt)
sf.write("nu_ref.wav", w_nu[0], sr_nu)

clear_output()
print("✅ Đã tạo xong 2 giọng mẫu!")
print("👨 Giọng Nam (nam_ref.wav):")
display(Audio("nam_ref.wav"))
print("👩 Giọng Nữ (nu_ref.wav):")
display(Audio("nu_ref.wav"))

In [2]:
# @title 🚀 BƯỚC 3: RENDER PODCAST SIÊU TỐC

# 👇👇👇 DÁN KỊCH BẢN CỦA BẠN VÀO ĐÂY (Giữa 3 dấu ngoặc kép) 👇👇👇
kich_ban = """
Gemini đã nói
Nam: Hello, everyone, and welcome back to the Just English Channel, the absolute best place on the internet to practice your English listening skills. I am your host, David, and I am currently enjoying the beautiful peace and quiet of an empty room.
Nu: And I am Emma. Welcome back, listeners. We are incredibly happy to have you with us today. David, why are you talking about an empty room? Are you hiding from someone?
Nam: I am not hiding, Emma. I am celebrating my personal space. Today, we are talking about family, specifically big families versus small families. And as an only child, I am the proud king of my own space.
Nu: That actually explains so much about your personality, David. Listeners, an only child is a person who has no brothers or sisters. I, on the other hand, come from a very big family. I am the oldest of six children.
Nam: Six children? Emma, that is not a family. That is a small military army. How did your parents survive the noise?
Nu: It was not an army, David, it was a home full of love and energy. Having siblings is a wonderful experience. For our A2 and B1 learners, a sibling is a brother or a sister. I have three sisters and two brothers. We did everything together.
Nam: Doing everything together sounds like a complete nightmare to me. Listeners, get ready for our first major debate of the day. The concept of sharing. Emma, as an only child, all the toys in the house were mine. All the television time was mine. And most importantly, all the snacks were mine.
Nu: Sharing is a basic human skill, David. In a big family, you learn to share your toys, your clothes, and your food. It teaches you to be generous and kind. You cannot be selfish. Selfish means you only care about yourself and what you want.
Nam: I am not selfish, Emma. I am highly protective of my pizza. If I order a large pepperoni pizza, I want to eat eight slices of pepperoni pizza. I do not want five other people reaching their hands into my beautiful pizza box. That is a crime against junk food.
Nu: Sharing a meal brings people closer together! In my house, if you wanted a slice of pizza, you had to be fast. But we always made sure everyone got a piece. It builds character. You learn to negotiate and compromise.
Nam: I do not want to negotiate for a slice of cheese, Emma. I just want to eat it while playing video games in total silence.
Nu: You see, listeners, this is the classic difference. Because David grew up alone, he loves his quiet time. In a big family, there is always chaos. Chaos means a state of total confusion with no order. But it is a happy chaos. Someone is always laughing, playing music, or telling a story.
Nam: Or screaming, crying, and breaking things. Do not lie to me, Emma. I have seen big families in the supermarket. It looks extremely stressful.
Nu: Well, yes, siblings do argue. We have a great English phrase for this. To drive someone crazy. It means to make someone feel very annoyed or frustrated. My little brothers used to drive me crazy when they entered my bedroom without knocking. But we always forgave each other.
Nam: I never had to forgive anyone because nobody ever entered my room. My bedroom was my personal castle. I had my bed, my computer, and a secret drawer full of chocolate candy. It was absolute perfection.
Nu: But didn't you ever feel lonely? When you are an only child, you do not have a built-in best friend living down the hall. Who did you play with on rainy days?
Nam: I played with my digital friends on the internet, Emma. Video game characters do not steal your clothes, they do not eat your chocolate, and you can simply turn them off when you are tired. It is the perfect relationship.
Nu: That is so sad and unnatural! Real human connection is important. In my family, we always had someone to talk to. If I was sad, I went to my sister's room. If I needed help with homework, my brother was there. We learned to get along with different personalities. To get along means to have a good, friendly relationship with someone.
Nam: I get along perfectly fine with my pizza delivery driver. We have a great relationship. He brings food, I say thank you, he leaves. No drama.
Nu: Listeners, here is our second heated debate for today. Let us talk about morning routines and organization. David, what was your morning routine like growing up in a small family?
Nam: Oh, it was majestic. Because it was just me and my parents, we had two bathrooms in the house. I could wake up at eleven o'clock in the morning on a Saturday. I could walk into the bathroom, take a warm forty-five-minute shower, and slowly get ready for the day. No rushing, no stress.
Nu: A forty-five-minute shower? That is a terrible waste of water! Let me describe a real morning routine in a big family. We had one bathroom for the six kids. One single bathroom.
Nam: That is not a home, Emma, that is a survival camp.
Nu: It required extreme discipline. I was the oldest, so I was the manager. I created a highly detailed Excel spreadsheet for the bathroom schedule. Every child had exactly seven minutes in the bathroom. If you stayed for eight minutes, the next person would loudly hit the door with a spoon until you came out.
Nam: You used an Excel spreadsheet to take a shower? Emma, you were a child! You were supposed to be watching cartoons, not running a corporate business.
Nu: Organization is the key to success, David. Because we were organized, we all brushed our teeth, ate a healthy breakfast of oatmeal and fruit, and walked to school perfectly on time. You were probably sleeping while I was managing five younger siblings.
Nam: I was definitely sleeping. And my mother was making me a beautiful plate of pancakes with extra syrup. I did not have to fight anyone for the syrup.
Nu: This is why you are so lazy now, David! You never had to fight for anything. In a big family, you learn responsibility very early. Every child had a chore. A chore is a small job you do around the house, like washing the dishes or taking out the trash. We had a chore chart on the refrigerator.
Nam: I had one chore growing up. My chore was to clean my own room once a week. Sometimes I just pushed all my dirty clothes under my bed and told my mother I was finished. I took a shortcut.
Nu: And your parents accepted that? Listeners, the phrase to take a shortcut means to find an easier or faster way to do something, usually by avoiding hard work. David is the king of taking shortcuts. My parents would never allow that. If we did not do our fair share of the work, we did not get to watch television.
Nam: To do your fair share. That means to do your part of the work equally with everyone else, right?
Nu: Exactly. Everyone must contribute. This teaches you how to be a good team player in the real world.
Nam: Well, laziness seems to run in my family. Listeners, to run in the family is a very useful idiom. It means a quality, ability, or physical feature that many people in the same family have. For example, my dad loves to sleep on the sofa, and I love to sleep on the sofa. Laziness runs in my family.
Nu: Laziness is a habit, David, not genetics. In my family, being active runs in the family. My parents wake up at five in the morning to run, and I wake up at five to go to the gym. We take after our parents.
Nam: To take after someone. That means to look or behave like an older member of your family. I definitely take after my dad. We both firmly believe that running is only necessary if a wild bear is chasing you.
Nu: You are unbelievable. But let us discuss another major difference. Family dinners. What was dinner time like in your small family?
Nam: Dinner was incredibly relaxing. The three of us would sit in the living room. We would turn on a great action movie on the television. We would order some delicious Chinese takeout or a massive pizza, and we would eat in comfortable silence while watching the explosions on the screen.
Nu: Eating on the sofa while watching television? That is terrible for your digestion! Listeners, prepare for debate number three. A family dinner should happen at a dining table.
Nam: A dining table is just a wooden desk for food. The sofa is soft and perfectly shaped for my body.
Nu: In a big family, dinner is the most important event of the day. My mother would cook a massive, healthy meal. A huge bowl of fresh salad, grilled chicken, and steamed vegetables. All eight of us would sit around a giant wooden table. There was no television allowed.
Nam: No television? What did you look at? Did you just stare at the broccoli?
Nu: We looked at each other, David! We talked. Everyone had to share one good thing that happened to them that day. It was loud, everyone was talking over each other, plates were passing across the table, and it was beautiful. You learn how to have a real conversation.
Nam: If eight people are talking at the same time, that is not a conversation. That is just noise. If I wanted that much noise, I would go to a crowded shopping mall.
Nu: It is a vibrant community. What about family vacations? Traveling with a big family is an unforgettable adventure. We had a massive, ugly minivan. My parents would pack the car the night before. We would wake up at four in the morning, squeeze into the seats like puzzle pieces, and drive for ten hours to go camping.
Nam: Ten hours in a car with five other children? Emma, I am feeling stressed just imagining that situation.
Nu: It was fun! We sang songs, we played road trip games, and we ate healthy carrot sticks.
Nam: Carrot sticks are not a road trip snack. Potato chips are a road trip snack. My small family vacations were completely different. We drove to the airport at a normal, reasonable time. We got on an airplane. I watched a movie on my tablet. And we went to a hotel with a swimming pool. I never had to share my hotel bed with a brother whose feet smelled bad.
Nu: You missed out on the adventure, David. The struggles of a big family trip become the best memories. One time, my brother forgot his shoes, and he had to wear my pink boots for the entire weekend. We still laugh about it today.
Nam: That actually sounds like a very funny story. In my family, I never wore anyone else's shoes. But I must admit, sometimes it was a little bit quiet in my house. Not bad, just quiet. I guess I am what they call the black sheep of the family, but a very lonely black sheep.
Nu: Oh, that is a fantastic idiom to teach our listeners. The black sheep. Listeners, the black sheep of the family is a person who is very different from the rest of their family, and sometimes the family disapproves of them. Are you really the black sheep, David?
Nam: Well, my parents are actually very neat and tidy people. They love cleaning. I prefer to let my clothes stay on the floor so I can easily see my fashion options. So yes, I am the lazy black sheep. What about you? Are there any lazy black sheep in your big family?
Nu: Absolutely not. My parents were very strict. If anyone tried to be lazy, the other siblings would force them to get up. The pressure to succeed was very high. Sometimes, in a big family, it is hard to get individual attention from your parents.
Nam: Finally, a negative point about the big family! See, as an only child, I had one hundred percent of my parents' attention. If I drew a terrible picture of a dog, they put it on the refrigerator and told me I was a genius.
Nu: Yes, you were very spoiled. Spoiled means a child is given everything they want and is never told no. That is why you think you deserve a whole pizza to yourself.
Nam: I do deserve a whole pizza to myself. I am a growing adult. But I think we have debated enough for today. Let us summarize the wonderful vocabulary and idioms we learned today before my brain runs out of energy.
Nu: Good idea. First, we learned about siblings, which are your brothers and sisters. And an only child, like David, who has no siblings and refuses to share his food.
Nam: We learned that if you are an only child, you never have to worry about someone driving you crazy, which means to deeply annoy or frustrate you. Like Emma's brothers walking into her room.
Nu: We also learned the phrasal verb to get along, which means to have a friendly relationship. In a big family, you must learn to get along with everyone.
Nam: Then we discussed chores, which are small jobs around the house. Emma had an Excel spreadsheet for her chores. I took a shortcut, which means to find an easy way to avoid hard work.
Nu: We learned the idiom to run in the family, meaning a quality that many family members share. And to take after someone, which means to behave or look like an older relative.
Nam: And finally, the black sheep. The person who is completely different from the rest of the family. Like a guy who loves junk food living in a house full of healthy salad eaters.
Nu: Excellent summary, David. Now, listeners, it is time for our viewer challenge! We want you to scroll down to the comments section right now and tell us about your family.
Nam: Yes! Are you proudly Team David? Are you an only child, or from a small family, who loves personal space, sleeping late, and keeping all the pizza for yourself?
Nu: Or are you Team Emma? Do you come from a big family where you had to share everything, use a bathroom schedule, and eat healthy meals at a noisy, happy dinner table? Tell us in the comments!
Nam: Be honest, listeners. We all know the quiet life of the small family is scientifically superior for your mental health.
Nu: Please ignore David's terrible science facts. We absolutely love reading your stories and practicing English with you all. If you enjoyed our funny conversation today, please hit the like button, subscribe to the Just English Channel, and turn on the notification bell.
Nam: Exactly. Hitting the subscribe button requires zero sharing and zero chores. It is the easiest shortcut on the internet.
Nu: Thank you so much for tuning in and practicing your listening skills with us. Keep studying, keep sharing your snacks, and we will see you in the next episode!
Nam: Goodbye everyone! I am going to go order a large pepperoni pizza right now, and I am going to eat the entire thing in a perfectly quiet, empty room.
Nu: You are completely hopeless, David. Go eat a carrot! Goodbye, listeners!
"""
# 👆👆👆 ========================================================== 👆👆👆

file_nam = "nam_ref.wav"
file_nu = "nu_ref.wav"

if not os.path.exists(file_nam) or not os.path.exists(file_nu):
    print("⚠️ LỖI: Không tìm thấy file giọng mẫu. Hãy chạy Bước 2 trước!")
else:
    model = load_model("CLONE")
    final_audio = AudioSegment.silent(duration=500)
    gap = AudioSegment.silent(duration=400)

    lines = [l for l in kich_ban.strip().split('\n') if ":" in l]
    total_lines = len(lines)

    print("⚡️ Đang học giọng Nam & Nữ (Chỉ làm 1 lần)...")
    nam_prompt_feature = model.create_voice_clone_prompt(ref_audio=file_nam, ref_text=None, x_vector_only_mode=True)
    nu_prompt_feature = model.create_voice_clone_prompt(ref_audio=file_nu, ref_text=None, x_vector_only_mode=True)

    print("🚀 BẮT ĐẦU THU ÂM PODCAST...")
    for index, line in enumerate(lines):
        name_part, text = line.split(":", 1)
        clean_name = re.sub(r'\(.*?\)', '', name_part).strip().lower()
        text = text.strip()

        # Nhận diện Nam/Nữ (Đã cập nhật đủ chữ david, emma)
        is_nam = clean_name in ["nam", "man", "male", "host", "teacher", "eric", "ryan", "mr", "david"]
        current_prompt = nam_prompt_feature if is_nam else nu_prompt_feature

        if text:
            print(f"🎙️ [{index+1}/{total_lines}] {clean_name.upper()}: Đang thu âm...")
            with torch.inference_mode():
                w, sr = model.generate_voice_clone(text=text, voice_clone_prompt=current_prompt)

            sf.write("temp.wav", w[0], sr)
            final_audio += AudioSegment.from_wav("temp.wav") + gap
            if os.path.exists("temp.wav"): os.remove("temp.wav")

    # Xuất file
    final_audio.export("Podcast_ThanhPham.mp3", format="mp3")
    clear_output()
    print("🎉 HOÀN TẤT! Đã ghép nối xong toàn bộ kịch bản.")
    print("👇 Bấm Play để nghe hoặc bấm dấu 3 chấm (⋮) để Tải xuống MP3 👇")
    display(Audio("Podcast_ThanhPham.mp3"))

🎉 HOÀN TẤT! Đã ghép nối xong toàn bộ kịch bản.
👇 Bấm Play để nghe hoặc bấm dấu 3 chấm (⋮) để Tải xuống MP3 👇
